In [9]:
# Cell 1 — Imports

from pathlib import Path
import yaml
import pandas as pd
import xarray as xr

In [10]:


PROJECT_ROOT = Path.cwd().parent
RUN_CONFIG = PROJECT_ROOT / "config.run.yaml"
print("Project root:", PROJECT_ROOT)
print("Run config:", RUN_CONFIG)
print("Exists:", RUN_CONFIG.exists())

Project root: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape
Run config: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/config.run.yaml
Exists: True


In [11]:
# Cell 3 — Load run configuration

with open(RUN_CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# cfg

In [12]:
# Cell 4 — Run scope summary

print("Region:", cfg["region"]["name"])
print("CRS:", cfg["region"]["crs"])
print("BBox:", cfg["region"]["bbox"])
print("Buffer (km):", cfg["region"]["buffer_km"])
print("Grid:", cfg["grid"]["system"], cfg["grid"]["resolution"])
print("Time:", cfg["time"]["start"], "->", cfg["time"]["end"])

Region: falkland_islands
CRS: EPSG:4326
BBox: {'xmin': -64, 'ymin': -57, 'xmax': -51, 'ymax': -47}
Buffer (km): 50
Grid: H3 6
Time: 2014-01-01 -> 2023-12-31


In [13]:
# Cell 6 — Discover raw datasets from data/raw

raw_root = PROJECT_ROOT / cfg["paths"]["raw"]

dataset_dirs = sorted([p for p in raw_root.iterdir() if p.is_dir()])

print("Raw root:", raw_root)
print("Datasets found:", len(dataset_dirs))

for d in dataset_dirs:
    print(" ", d.name)

Raw root: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw
Datasets found: 5
  chl
  gfw
  ssh
  sst
  wind


In [14]:
# Cell 7 — Discover raw datasets and files

raw_root = PROJECT_ROOT / cfg["paths"]["raw"]

dataset_dirs = sorted([p for p in raw_root.iterdir() if p.is_dir()])

datasets = {}

for dataset_dir in dataset_dirs:
    files = sorted(
        [
            p for p in dataset_dir.rglob("*")
            if p.is_file() and not p.name.startswith(".")
        ]
    )
    datasets[dataset_dir.name] = {
        "directory": dataset_dir,
        "files": files,
    }

print("Raw root:", raw_root)
print("Datasets found:", len(datasets))

for name, meta in datasets.items():
    print(f"  {name}: {len(meta['files'])} files")

Raw root: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw
Datasets found: 5
  chl: 1 files
  gfw: 10 files
  ssh: 1 files
  sst: 3652 files
  wind: 120 files


In [15]:
# Cell 8 — Inspect dataset organization

from collections import Counter

def summarize_dataset_files(files):
    suffixes = Counter(p.suffix.lower() for p in files)
    return {
        "file_count": len(files),
        "suffixes": dict(suffixes),
        "storage_pattern": "single-file dataset" if len(files) == 1 else "multi-file dataset",
        "first_file": files[0].name if files else None,
        "last_file": files[-1].name if len(files) > 1 else None,
    }

def print_dataset_summary(name, directory, summary):
    print(f"Dataset: {name}")
    print(f"Directory: {directory}")
    if summary["file_count"] == 0:
        print("Note: No files found")
        return

    print(f"File count: {summary['file_count']}")
    print("File types:", summary['suffixes'])
    print(f"Storage pattern: {summary['storage_pattern']}")
    print(f"First file: {summary['first_file']}")
    if summary["last_file"] is not None:
        print(f"Last file: {summary['last_file']}")
    print()

for name, meta in datasets.items():
    files = [p for p in meta['files'] if not p.name.startswith('.') ]
    print_dataset_summary(name, meta['directory'], summarize_dataset_files(files))


Dataset: chl
Directory: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw/chl
File count: 1
File types: {'.nc': 1}
Storage pattern: single-file dataset
First file: cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D_CHL_64.73W-50.27W_57.44S-46.56S_2014-01-01-2023-12-31.nc

Dataset: gfw
Directory: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw/gfw
File count: 10
File types: {'.parquet': 10}
Storage pattern: multi-file dataset
First file: fishing_effort.parquet
Last file: fishing_effort.parquet

Dataset: ssh
Directory: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw/ssh
File count: 1
File types: {'.nc': 1}
Storage pattern: single-file dataset
First file: cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt_64.69W-50.31W_57.44S-46.56S_2014-01-01-2023-12-31.nc

Dataset: sst
Directory: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw/sst
File count: 3652
File types: {'.nc': 3652}
Storage pattern: multi-file dataset
First fi

In [16]:
# Cell 9 — Inspect one sample file per dataset (improved formatting)

import tempfile
import zipfile

NC_SUFFIXES = {".nc", ".nc4", ".cdf"}

def summarize_xarray_dataset(path):
    """Return a compact summary of an xarray-readable dataset."""
    with xr.open_dataset(path) as ds:
        time_coverage = None
        for coord_name, coord in ds.coords.items():
            if str(coord.dtype).startswith("datetime64"):
                time_coverage = (
                    coord_name,
                    str(coord.min().values),
                    str(coord.max().values),
                )
                break

        return {
            "dimensions": dict(ds.sizes),
            "variables": list(ds.data_vars),
            "time_coverage": time_coverage,
        }

def inspect_sample_file(path):
    """Inspect one file or archive and return a compact summary."""
    suffix = path.suffix.lower()
    info = {
        "sample_file": path.name,
        "container": suffix.lstrip("."),
        "summary": None,
        "members": None,
        "note": None,
    }

    if suffix in NC_SUFFIXES:
        info["summary"] = summarize_xarray_dataset(path)
        return info

    if suffix == ".zip":
        with zipfile.ZipFile(path, "r") as z:
            members = [
                member
                for member in z.namelist()
                if member and not Path(member).name.startswith('.')
            ]
            nc_members = [
                member
                for member in members
                if Path(member).suffix.lower() in NC_SUFFIXES
            ]

            if not nc_members:
                info["note"] = "ZIP contains no NetCDF file"
                return info

            summaries = []
            with tempfile.TemporaryDirectory() as tmpdir:
                for member in nc_members:
                    extracted = Path(z.extract(member, path=tmpdir))
                    member_summary = summarize_xarray_dataset(extracted)
                    member_summary["container"] = Path(member).suffix.lstrip(".").lower()
                    summaries.append({
                        "member": member,
                        **member_summary,
                    })

            info["members"] = summaries
            info["note"] = (
                f"ZIP archive with {len(members)} members; "
                f"{len(nc_members)} NetCDF datasets inspected"
            )
            return info

    info["note"] = f"Unsupported file type: {suffix}"
    return info

def print_summary(summary, indent=""):
    print(f"{indent}Dimensions: {summary['dimensions']}")
    print(f"{indent}Variables: {summary['variables']}")
    if summary["time_coverage"] is not None:
        coord_name, start, end = summary["time_coverage"]
        print(f"{indent}Time coverage ({coord_name}): {start} -> {end}")

for name, meta in datasets.items():
    files = [p for p in meta["files"] if not p.name.startswith('.')]

    print(f"Dataset: {name}")

    if not files:
        print("Note: No files found")
        print()
        continue

    info = inspect_sample_file(files[0])
    print(f"Sample file: {info['sample_file']}")
    print(f"Container: {info['container']}")

    if info["summary"] is not None:
        print_summary(info["summary"])

    if info["members"] is not None:
        for member_info in info["members"]:
            print(f"    Sample file: {member_info['member']}")
            print(f"    Container: {member_info["container"]}")
            print_summary(member_info, indent="    ")
            print()

    if info["note"]:
        print(f"Note: {info['note']}")
    print()


Dataset: chl
Sample file: cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D_CHL_64.73W-50.27W_57.44S-46.56S_2014-01-01-2023-12-31.nc
Container: nc
Dimensions: {'time': 3652, 'latitude': 262, 'longitude': 348}
Variables: ['CHL']
Time coverage (time): 2014-01-01T00:00:00.000000000 -> 2023-12-31T00:00:00.000000000

Dataset: gfw
Sample file: fishing_effort.parquet
Container: parquet
Note: Unsupported file type: .parquet

Dataset: ssh
Sample file: cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt_64.69W-50.31W_57.44S-46.56S_2014-01-01-2023-12-31.nc
Container: nc
Dimensions: {'time': 3652, 'latitude': 88, 'longitude': 116}
Variables: ['adt']
Time coverage (time): 2014-01-01T00:00:00.000000000 -> 2023-12-31T00:00:00.000000000

Dataset: sst
Sample file: sst_20140101.nc
Container: nc
Dimensions: {'time': 1, 'lat': 1091, 'lon': 1447}
Variables: ['analysed_sst']
Time coverage (time): 2014-01-01T09:00:00.000000000 -> 2014-01-01T09:00:00.000000000

Dataset: wind
Sample file: deri